# 🎬 Estúdio — Kaggle (alternativa ao Colab, 2x T4)

Em **Settings**: **Accelerator → GPU T4 x2** e **Internet → On**. Para o site, crie o segredo
`HF_TOKEN` em **Add-ons → Secrets**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
RAMO = "estudio-video"
!rm -rf Jarvis && git clone -q -b {RAMO} --depth 1 https://github.com/gutodrac-spec/Jarvis.git
%cd Jarvis/estudio
!pip install -q -r requirements.txt

In [ ]:
# 3. Site fixo no Hugging Face (opcional, mas recomendado).
#    Cria uma vez o Space estático https://huggingface.co/spaces/<seu-usuario>/estudio com a página.
#    Toda vez que o app subir, ele grava ali o link novo, e o site se conecta sozinho.
import os
from huggingface_hub import HfApi

NOME_DO_SITE = "estudio"
from kaggle_secrets import UserSecretsClient
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    api = HfApi(token=HF_TOKEN)
    SITE = f"{api.whoami()['name']}/{NOME_DO_SITE}"
    api.create_repo(SITE, repo_type="space", space_sdk="static", exist_ok=True)
    api.upload_file(repo_id=SITE, repo_type="space", path_in_repo="README.md", path_or_fileobj=(
        "---\ntitle: Estúdio\nemoji: 🎬\nsdk: static\npinned: false\n---\n\n"
        "Gerador de vídeo com presets de câmera. A GPU roda no Google Colab.\n").encode())
    api.upload_file(repo_id=SITE, repo_type="space", path_in_repo="index.html", path_or_fileobj="web/index.html")
    os.environ["ESTUDIO_SITE"] = SITE
    print("Site:", f"https://huggingface.co/spaces/{SITE}")
else:
    print("Sem HF_TOKEN: pulei o site. Use o link gradio.live direto.")

In [ ]:
# 4. Pré-carrega o modelo (baixado do Hugging Face, ~16 GB na 1ª vez) e sobe a interface + API.
#    Deixe esta célula rodando: enquanto ela estiver ativa, o site funciona.
!ESTUDIO_SHARE=1 ESTUDIO_PRECARREGAR=1 python app.py